# Voice of Customer Intelligence com RAG — resumo executivo

Este notebook percorre o projeto chamando os módulos de `src/`. A lógica vive nos módulos (versionáveis e testados); aqui ficam só a leitura guiada dos resultados.

Pré-requisito: `python main.py` já executado (camadas bronze, silver e gold + índice vetorial). As seções 4 e 5 chamam a OpenAI (custo de fração de centavo por pergunta).

In [ ]:
import sys, json
sys.path.insert(0, '..')
import pandas as pd
from IPython.display import Image, Markdown, display
from src import config

## 1. Camadas da base de conhecimento

In [ ]:
manifesto = json.loads(config.MANIFESTO_BRONZE.read_text(encoding='utf-8'))
silver = pd.read_parquet(config.SILVER_AVALIACOES)
gold = pd.read_parquet(config.GOLD_DOCUMENTOS)
display(pd.DataFrame([
    {'camada': 'bronze', 'conteúdo': 'avaliações (CSV original)', 'linhas': manifesto['arquivos']['olist_order_reviews_dataset.csv']['linhas']},
    {'camada': 'silver', 'conteúdo': 'avaliações únicas, limpas e enriquecidas', 'linhas': len(silver)},
    {'camada': 'silver', 'conteúdo': '... com comentário', 'linhas': int(silver['has_text'].sum())},
    {'camada': 'gold', 'conteúdo': 'documentos indexados no RAG', 'linhas': len(gold)},
]))
gold[['review_id', 'review_score', 'delivery_status', 'product_category', 'text']].head()

## 2. Análise exploratória

In [ ]:
display(Markdown((config.REPORTS_DIR / 'eda.md').read_text(encoding='utf-8')))

## 3. Recuperação híbrida (sem LLM)

Busca densa (ChromaDB) + BM25, fundidas por Reciprocal Rank Fusion, com filtro de metadados.

In [ ]:
from src.rag.recuperacao import get_retriever
from src.rag.schemas import SearchFilters

retriever = get_retriever()
resultados = retriever.search(['o produto não chegou até hoje', 'a entrega atrasou muito'],
                              SearchFilters(score_max=2), top_k=8)
pd.DataFrame([{'nota': r.metadata['review_score'], 'entrega': r.metadata['delivery_status'],
               'cosseno': round(r.dense_similarity or 0, 3), 'rank_bm25': r.best_bm25_rank,
               'rrf': round(r.rrf_score, 4), 'texto': r.text} for r in resultados])

## 4. Pipeline completo (LangGraph + OpenAI)

In [ ]:
from src.rag.grafo import VoCAssistant
from src.rag.formatacao import render_state

assistente = VoCAssistant(retriever=retriever)
estado = assistente.ask('Quais padrões aparecem nas avaliações de clientes insatisfeitos?')
display(Markdown(render_state(estado)))

### Perguntas sem suporte nos dados

In [ ]:
for pergunta in ['Qual é a previsão do tempo para amanhã em São Paulo?',
                 'Quais reclamações os clientes fazem sobre pagamentos com criptomoedas?']:
    estado = assistente.ask(pergunta)
    print(pergunta)
    print(f"  -> {estado['status']}: {estado['message'][:120]}...")
    print()

## 5. Avaliação

Resultado de `python main.py --etapa avaliacao`.

In [ ]:
display(Image(str(config.IMAGES_DIR / '08_ablacao_recuperacao.png')))
display(Markdown((config.REPORTS_DIR / 'avaliacao.md').read_text(encoding='utf-8').split('## 3.')[0].split('## 2.')[1]))